In [ ]:
# התקנת הספריות הנדרשות
!pip install -q transformers datasets evaluate accelerate scikit-learn

import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
from google.colab import userdata
from huggingface_hub import login
import os
import shutil


print("מוריד את דטאסט PlantVillage הרשמי...")
!wget -q https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
!unzip -q master.zip

print("מסנן את התמונות ומכין תיקיית עגבניות בלבד...")
source_dir = "PlantVillage-Dataset-master/raw/color"
target_dir = "tomato_dataset"

# מחיקת התיקייה אם היא כבר קיימת כדי למנוע כפילויות
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
os.makedirs(target_dir)

# העתקת רק תיקיות של עגבנייה
for folder_name in os.listdir(source_dir):
    if folder_name.startswith("Tomato"):
        src_path = os.path.join(source_dir, folder_name)
        dst_path = os.path.join(target_dir, folder_name)
        shutil.copytree(src_path, dst_path)

print("טוען את התמונות למבנה נתונים...")
# טעינה באמצעות imagefolder שיוצרת את התוויות אוטומטית
dataset = load_dataset("imagefolder", data_dir=target_dir)

# חילוץ שמות המחלקות ויצירת מיפוי למספרים
label_names = dataset['train'].features['label'].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

# חלוקה לאימון, ולידציה ובדיקה
splits = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_ds = splits['train']
val_test_splits = splits['test'].train_test_split(test_size=0.5, seed=42)
val_ds = val_test_splits['train']
test_ds = val_test_splits['test']

# עיבוד מקדים לתמונות
model_name = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)

def transforms(examples):
    inputs = processor([img.convert("RGB") for img in examples["image"]], return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

train_ds.set_transform(transforms)
val_ds.set_transform(transforms)
test_ds.set_transform(transforms)

# הגדרת המודל
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

# הגדרות האימון
training_args = TrainingArguments(
    output_dir="./tomadoc_model",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    num_train_epochs=5,
    fp16=True,
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn
)

# התחלת אימון
print("מתחיל אימון...")
trainer.train()

# בדיקת ביצועים סופית
print("מעריך ביצועים על נתוני הבדיקה...")
test_results = trainer.evaluate(test_ds)
print(f"דיוק סופי: {test_results['eval_accuracy']:.4f}")


# משיכת המפתח מהסודות של קולאב והתחברות
print("מושך את המפתח ומבצע התחברות...")
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

local_directory = "./tomadoc_vit_model"
hf_repository_name = "tomadoc-disease-classifier"

# שמירה מקומית של המודל והמעבד
print(f"שומר את המודל מקומית בנתיב: {local_directory}")
trainer.save_model(local_directory)
processor.save_pretrained(local_directory)

# יצירת תוכן מובנה עבור כרטיס המודל (Model Card)
accuracy_score = test_results.get('eval_accuracy', 0.0)

model_card_content = f"""---
language:
- en
license: mit
tags:
- image-classification
- vit
- agriculture
- tomato-diseases
datasets:
- nateraw/plant-village
metrics:
- accuracy
pipeline_tag: image-classification
---

# Tomadoc Tomato Disease Classifier

This model is a fine-tuned version of `google/vit-base-patch16-224` optimized for identifying diseases in tomato plant leaves. It was trained to serve as the backbone core for the Tomadoc application.

## Model Description
- **Model Type:** Vision Transformer (ViT)
- **Task:** Image Classification (10 Tomato classes)
- **Base Model:** google/vit-base-patch16-224
- **Dataset:** PlantVillage (Tomato leaf subset containing healthy and diseased leaves)

## Training & Hardware Details
- **Hardware Used:** NVIDIA T4 GPU (Google Colab)
- **Precision:** Mixed Precision (FP16)
- **Learning Rate:** 2e-5
- **Batch Size:** 32
- **Epochs:** 5

## Intended Uses & Limitations
This model is intended to classify leaf images of tomato plants into their respective healthy or diseased categories. It performs best when provided with clear, centered images of single leaves under decent lighting.

## Evaluation Results
The model was evaluated on a dedicated test split from the filtered dataset:
- **Final Test Accuracy:** {accuracy_score:.4f}
"""

# כתיבת קובץ ה-README לתיקיית המודל
readme_path = os.path.join(local_directory, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card_content)
print("כרטיס המודל נוצר בהצלחה בתיקייה המקומית.")


מוריד את דטאסט PlantVillage הרשמי...
מסנן את התמונות ומכין תיקיית עגבניות בלבד...
טוען את התמונות למבנה נתונים...


Resolving data files:   0%|          | 0/18160 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

...

דיוק סופי: 0.9961
מושך את המפתח ומבצע התחברות...
שומר את המודל מקומית בנתיב: ./tomadoc_vit_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

כרטיס המודל נוצר בהצלחה בתיקייה המקומית.
מעלה את כל הקבצים למאגר: tomadoc-disease-classifier


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TypeError: Trainer.create_model_card() got an unexpected keyword argument 'repo_id'

In [ ]:
from huggingface_hub import HfApi

# הגדרת הנתיבים (נוודא שהם זהים למה שהגדרנו קודם)
local_directory = "./tomadoc_vit_model"
hf_repository_name = "oshriagronov/tomadoc-mythos"

api = HfApi()

# יצירת המאגר בענן למקרה שהוא עדיין לא קיים
print("מוודא שהמאגר קיים בחשבון שלך...")
api.create_repo(repo_id=hf_repository_name, exist_ok=True)

# העלאת כל התוכן של התיקייה
print("מעלה את המודל, המעבד וכרטיס המודל לענן...")
api.upload_folder(
    folder_path=local_directory,
    repo_id=hf_repository_name,
    commit_message="Upload trained Tomadoc model and files"
)

print("ההעלאה הושלמה בהצלחה!")

מוודא שהמאגר קיים בחשבון שלך...
מעלה את המודל, המעבד וכרטיס המודל לענן...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t_model/model.safetensors:   0%|          |  551kB /  343MB            

  ...t_model/training_args.bin:   7%|6         |   356B / 5.14kB            

ההעלאה הושלמה בהצלחה!
